# ESM3/C

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
os.chdir(os.path.dirname(os.path.abspath('.')))

import src.utils as utils
import src.config as config
import src.haplosaurus as hs

# import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

## ESM3

In [2]:
import torch
available_gpus = [torch.cuda.device(i) for i in range(torch.cuda.device_count())]
available_gpus

In [4]:
devices = vm.get_available_gpus()
devices

[(0, 84974239744), (1, 84974239744), (2, 84974239744), (3, 84974239744)]

In [13]:
import torch
devices = []
for i in range(torch.cuda.device_count()):
   devices.append(torch.cuda.get_device_properties(i))
devices

[_CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=57a5dbce-e7ee-766b-1803-e35688317a0c, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ee601dc3-6f2c-3e1b-8b01-5c4012c77075, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=d1d17271-1e44-abb0-9cf6-de92acf10943, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ca14fd86-8f12-0e67-15c5-eefe075aa2dd, L2_cache_size=40MB)]

In [31]:
import torch
torch.cuda.set_device(2)

In [16]:
from huggingface_hub import login   
from esm.models.esm3 import ESM3
from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig


# Will instruct you how to get an API key from huggingface hub, make one with "Read" permission.
# login( token="{HUGGINGFACE_TOKEN}")

# This will download the model weights and instantiate the model on your machine.
model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("cuda")  # Use GPU 2 which has the most available memory

# # Generate a completion for a partial Carbonic Anhydrase (2vvb)
prompt = "___________________________________________________DQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP___________________________________________________________"
protein = ESMProtein(sequence=prompt)
# Generate the sequence, then the structure. This will iteratively unmask the sequence track.
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8, temperature=0.7))
# We can show the predicted structure for the generated sequence.
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./generation.pdb")
# Then we can do a round trip design by inverse folding the sequence and recomputing the structure
protein.sequence = None
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8))
protein.coordinates = None
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./round_tripped.pdb")

100%|██████████| 8/8 [00:00<00:00, 21.77it/s]
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/utils/structure/protein_complex.py:223: UserWarning: Entity ID not found in metadata, using None as default
  warnings.warn("Entity ID not found in metadata, using None as default")
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/models/vqvae.py:286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: ignore
100%|██████████| 8/8 [00:00<00:00, 22.21it/s]


In [22]:
protein

ESMProtein(sequence='SAATKDSMAKMAYKPSAADTDHWPQWMTPLTGRVKTCAPGISYRRPIKFGYGDANTANLENKGNGPTPDLVSKGDGTVVSGAFLPYPYAAIQLHLHWSTDVNHGGEHSINGKHYAAELHIVHWNTKYGSVGEAITHDNGLAVLGIFIEIGRENKNLQRLTGLLDSIRRKGDTVDLKSFNLQDFFPDSLNYYTYKGSLTTPPCKGSVIWTSFNKPIEISPQQIADVQKKVEKEGNGKMNINNFRDRFAQRLDGRVLSIDML', secondary_structure=None, sasa=None, function_annotations=None, coordinates=tensor([[[-11.9912,  -7.3935,  25.6203],
         [-10.9490,  -6.9068,  24.7224],
         [ -9.8503,  -6.1854,  25.4958],
         ...,
         [     inf,      inf,      inf],
         [     inf,      inf,      inf],
         [     inf,      inf,      inf]],

        [[ -9.5176,  -7.0718,  25.6691],
         [ -8.2840,  -6.3995,  26.0633],
         [ -7.1847,  -6.6178,  25.0291],
         ...,
         [     inf,      inf,      inf],
         [     inf,      inf,      inf],
         [     inf,      inf,      inf]],

        [[ -7.4287,  -6.7658,  24.8261],
         [ -6.1664,  -6.8051,  24.0952],
         [ -5.0198,  -7.2398,  25.

## ESMC

In [40]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

protein = ESMProtein(sequence="AAAAA")

client = ESMC.from_pretrained("esmc_300m").to("cuda") # or "cpu"
protein_tensor = client.encode(protein)
logits_output = client.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.logits, logits_output.embeddings)

ForwardTrackData(sequence=tensor([[[-38.2500, -38.0000, -38.0000,  12.5625,  21.6250,  22.2500,  22.0000,
           21.8750,  21.5000,  21.6250,  21.7500,  21.3750,  20.7500,  21.5000,
           22.0000,  21.0000,  20.7500,  20.3750,  20.3750,  20.2500,  21.8750,
           19.8750,  19.8750,  19.3750,  18.3750,   1.1875,  -1.7812,  -4.1250,
          -20.7500, -38.0000, -38.0000, -38.2500, -38.0000, -38.2500, -38.2500,
          -38.2500, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.2500, -38.2500, -38.0000, -38.2500,
          -38.0000, -38.2500, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000],
         [-40.0000, -40.0000, -40.0000,   5.4062,  20.1250,  19.8750,  18.7500,
           20.6250,  18.8750,  18.3750,  18.6250,  18.5000,  18.1250,  18.0000,
           18.7500,  17.7500,  17.3750,  17.1250,  17.7500,  16.8750,  22

In [41]:
print(logits_output.logits.sequence.shape)
print(logits_output.embeddings.shape)

torch.Size([1, 7, 64])
torch.Size([1, 7, 960])
